# llm.c d16 (~254M params) on 5.4B ClimbMix tokens, Colab A100

Fork of karpathy/llm.c: https://github.com/michaelp91-dev/llm.c

* Data is tokenized **in the background**: training starts as soon as the val shard and the first train shard exist.
  Every finished shard is also saved to a private HF dataset repo, so a rerun downloads them instead of re-tokenizing.
* Every checkpoint (every 250 steps, ~35 min) is pushed to the private model repo in the background (`ckpt/`), only the latest is kept.
* After a disconnect: **Runtime -> Run all** again. It re-downloads the data, pulls the latest checkpoint and resumes (`-y 1`).
* At the end it exports to Hugging Face format, uploads it to the repo root and disconnects the runtime. Nothing goes to Google Drive.

Before running: add a Colab secret named `HF_TOKEN` (key icon on the left, write access, notebook access on), and pick an A100 runtime.

In [ ]:
# 1. config
import os, glob, re, time, subprocess
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # read by huggingface_hub, never printed

HF_USER        = "michaelp91-dev"
MODEL_REPO     = f"{HF_USER}/llmc-d16"                  # checkpoints in ckpt/, final HF export at the root
DATA_REPO      = f"{HF_USER}/llmc-climbmix-gpt2-5.4B"   # tokenized shards (set to None to skip)
TARGET_TOKENS  = 5_400_000_000
SHARD_TOKENS   = 120_000_000
VAL_TOKENS     = 10_000_000
TOTAL_BATCH    = 524_288                                # -d, tokens per optimizer step
NUM_SHARDS     = -(-TARGET_TOKENS // SHARD_TOKENS)      # 45
MAX_STEPS      = TARGET_TOKENS // TOTAL_BATCH           # 10,299 (~1 epoch)
OUT            = "/content/models/d16"
HF_EXPORT      = "/content/models/d16-hf"
DATA           = "/content/llm.c/dev/data/climbmix"
UNASSIGN_ON_FAILURE = True   # disconnect (stop paying) if training crashes; checkpoints are already on HF
print(f"{NUM_SHARDS} shards, {MAX_STEPS} steps")

In [ ]:
# 2. get the fork, the starter pack (GPT-2 tokenizer + HellaSwag), deps, and build
%cd /content
if not os.path.isdir("/content/llm.c"):
    !git clone https://github.com/michaelp91-dev/llm.c.git
else:
    !cd /content/llm.c && git pull --ff-only
%cd /content/llm.c
!pip install -q tiktoken datasets huggingface_hub
if not os.path.exists("gpt2_tokenizer.bin") or not os.path.exists("dev/data/hellaswag/hellaswag_val.bin"):
    !chmod u+x ./dev/download_starter_pack.sh && ./dev/download_starter_pack.sh
!make train_gpt2cu
assert os.path.exists("train_gpt2cu"), "build failed"

In [ ]:
# 3. tokenize in the background (or download shards already in DATA_REPO, then continue)
hub = f"--hf_repo {DATA_REPO}" if DATA_REPO else ""
!nohup python dev/data/climbmix_stream.py -o {DATA} -n {TARGET_TOKENS} -s {SHARD_TOKENS} -v {VAL_TOKENS} {hub} > /content/data_prep.log 2>&1 &
print("data prep started, log: /content/data_prep.log")

In [ ]:
# 4. wait for the val shard and the first train shard
def producer_running():
    return subprocess.run(["pgrep", "-f", "climbmix_stream.py"], capture_output=True).returncode == 0
need = [f"{DATA}/climbmix_val_000000.bin", f"{DATA}/climbmix_train_000000.bin"]
while not all(os.path.exists(p) for p in need):
    if not producer_running() and not os.path.exists(f"{DATA}/STREAM_DONE"):
        !tail -n 30 /content/data_prep.log
        raise RuntimeError("data prep stopped, see the log above")
    time.sleep(15)
!tail -n 5 /content/data_prep.log
print("ready:", len(glob.glob(f"{DATA}/climbmix_train_*.bin")), "train shard(s) so far")

In [ ]:
# 5. resume support: pull the latest checkpoint from the model repo (no-op on a fresh start)
!python dev/hf_pull.py --repo {MODEL_REPO} --out_dir {OUT}

In [ ]:
# 6. train d16 (GPT-2 recipe). -W 1 waits for shards still being tokenized, -hf pushes checkpoints in the background
os.makedirs(OUT, exist_ok=True)
cmd = (f'./train_gpt2cu -i "{DATA}/climbmix_train_*.bin" -j "{DATA}/climbmix_val_*.bin" '
       f'-e d16 -b 16 -t 1024 -d {TOTAL_BATCH} -l 0.0006 -u 700 -q 0.0 -c 0.1 '
       f'-v 250 -s 1000 -h 1 -o {OUT} -n 250 -nk 2 -y 1 '
       f'-x {MAX_STEPS} -W 1 -WN {NUM_SHARDS} -hf {MODEL_REPO}')
print(cmd)
with open(f"{OUT}/train_stdout.log", "a") as logf:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
    rc = p.wait()

def last_step():
    steps = [int(m.group(1)) for f in glob.glob(f"{OUT}/DONE_*") if (m := re.search(r"DONE_(\d+)$", f))]
    return max(steps, default=-1)

if rc != 0 or last_step() != MAX_STEPS:
    print(f"training stopped early (exit code {rc}, last checkpoint {last_step()})")
    !python dev/hf_push.py --wait {OUT} --repo {MODEL_REPO}
    try:  # keep the console output with the checkpoints
        from huggingface_hub import HfApi
        HfApi().create_repo(MODEL_REPO, private=True, exist_ok=True)
        HfApi().upload_file(path_or_fileobj=f"{OUT}/train_stdout.log", path_in_repo="train_stdout.log", repo_id=MODEL_REPO)
    except Exception as e:
        print("could not upload train_stdout.log:", e)
    if UNASSIGN_ON_FAILURE:
        from google.colab import runtime
        runtime.unassign()
    raise RuntimeError("training did not finish; Run all again to resume")

In [ ]:
# 7. make sure the final checkpoint push has finished
!python dev/hf_push.py --wait {OUT} --repo {MODEL_REPO}

In [ ]:
# 8. export the final model to Hugging Face format and upload it to the repo root
step = last_step()
!python dev/eval/export_hf.py --input {OUT}/model_{step:08d}.bin --output {HF_EXPORT}
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path=HF_EXPORT, repo_id=MODEL_REPO, commit_message=f"HF export of step {step}")
api.upload_file(path_or_fileobj=f"{OUT}/train_stdout.log", path_in_repo="train_stdout.log", repo_id=MODEL_REPO)
print(f"done: https://huggingface.co/{MODEL_REPO}")

In [ ]:
# 9. disconnect so the runtime stops using compute units
from google.colab import runtime
runtime.unassign()